# 演習10 解答編 ―― 壊さないように測る

## 発展課題1 の解答 ―― 3段パイプラインにしたら

### (1) 予測

10-4 の表から、そのまま出ます。

```
一番遅い段は show の 33.1ms
上限A = 1000 / 33.1 = 30.2 FPS
上限B = 67.6 FPS（まだ遠い）
```

直列は 16.7 FPS でした。**30 FPS 前後まで伸びるはず**です。

「待ち」の段が最大だったので、**段を分ければ Read と Infer の裏に隠せます。**
逆に言えば、`show` の 33.1ms そのものは1ミリも短くなりません。
**隠せるだけです。だから 30.2 FPS を超えることはありません。**

### (2) 測る

段ごとに3つの数字を足し込みます。**各スレッドが自分専用に持つので、鍵は要りません**（10-2-2 ②）。

```cpp
TP a = now_(); q1.pop();   inf.win  += us_since(a);   // 取り出せるまで待った
TP b = now_(); do_infer(); inf.busy += us_since(b);   // 正味
TP c = now_(); q2.push(i); inf.wout += us_since(c);   // 入れられるまで待った
```

In [ ]:
%%writefile ans10a.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
#include <cstdlib>
using namespace std::chrono;

using TP = steady_clock::time_point;
static inline TP now_() { return steady_clock::now(); }

// ---- 演習5で組み立てたキュー（中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t cap) : cap_(cap) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < cap_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_; std::size_t cap_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

// ---- 仕事の中身。ex10c.cpp とまったく同じ ----
long calib = 0;
volatile long sink = 0;
long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }
void cpu_ms(int ms)  { sink += burn(calib * ms); }
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }
void calibrate() {
    long n = 100000;
    for (;;) {
        TP t = now_(); sink += burn(n);
        auto us = duration_cast<microseconds>(now_() - t).count();
        if (us > 30000) { calib = n * 1000 / us; break; }
        n *= 2;
    }
}
double effective_cores(unsigned hw) {
    long U = calib * 60;
    TP t = now_(); sink += burn(U);
    double one = duration_cast<microseconds>(now_() - t).count();
    std::vector<std::thread> th;
    t = now_();
    for (unsigned k = 0; k < hw; k++) th.emplace_back([&] { sink += burn(U); });
    for (auto& x : th) x.join();
    return hw * one / duration_cast<microseconds>(now_() - t).count();
}

const int N = 30;
int SHOW_MS = 30;                       // 第1引数で変える（発展課題2）

void do_read()      { cpu_ms(13); }
void do_infer()     { cpu_ms(4); cpu_ms(5); wait_ms(2); cpu_ms(2); }   // pre + dpu + post
void do_show(int i) { wait_ms(i % 10 == 0 ? SHOW_MS * 2 : SHOW_MS); }

// 1つの段について測る3つの数字。各スレッドが自分専用に持つので、鍵は要らない
struct Stat { long long busy = 0, win = 0, wout = 0; };
long long us_since(TP t) { return duration_cast<microseconds>(now_() - t).count(); }
double ms(long long us)  { return us / 1000.0 / N; }

int main(int argc, char** argv) {
    if (argc >= 2) SHOW_MS = atoi(argv[1]);
    calibrate();
    double eff = effective_cores(std::thread::hardware_concurrency());
    const double CPU_TOTAL = 13 + 4 + 5 + 2;        // Read + pre + post（dpu と show は「待ち」）

    BoundedQueue<int> q1(4), q2(4);
    Stat rd, inf, sh;
    TP t0 = now_();

    std::thread reader([&] {
        for (int i = 0; i < N; i++) {
            TP a = now_(); do_read();  rd.busy += us_since(a);
            TP b = now_(); q1.push(i); rd.wout += us_since(b);
        }
    });
    std::thread inferer([&] {
        for (int i = 0; i < N; i++) {
            TP a = now_(); q1.pop();   inf.win  += us_since(a);
            TP b = now_(); do_infer(); inf.busy += us_since(b);
            TP c = now_(); q2.push(i); inf.wout += us_since(c);
        }
    });
    std::thread shower([&] {
        for (int i = 0; i < N; i++) {
            TP a = now_(); q2.pop();    sh.win  += us_since(a);
            TP b = now_(); do_show(i);  sh.busy += us_since(b);
        }
    });
    reader.join(); inferer.join(); shower.join();
    double sec = us_since(t0) / 1e6;

    const char* nm[3]   = {"Read", "Infer", "Show"};
    double busy[3] = {ms(rd.busy), ms(inf.busy), ms(sh.busy)};
    double win [3] = {0.0,         ms(inf.win),  ms(sh.win)};
    double wout[3] = {ms(rd.wout), ms(inf.wout), 0.0};

    int culprit = 0;                    // 待ちの合計がいちばん小さい段 ＝ 犯人
    for (int k = 1; k < 3; k++)
        if (win[k] + wout[k] < win[culprit] + wout[culprit]) culprit = k;

    std::cout << std::fixed << std::setprecision(1)
              << "Show = " << SHOW_MS << "ms（10枚に1枚は " << SHOW_MS * 2 << "ms）\n\n"
              << "     正味  取り出し待ち  入れ待ち   段\n";
    for (int k = 0; k < 3; k++)
        std::cout << std::setw(8) << busy[k] << "ms" << std::setw(11) << win[k] << "ms"
                  << std::setw(9) << wout[k] << "ms   " << nm[k]
                  << (k == culprit ? "   <- 待っていない ＝ ここが犯人" : "") << "\n";
    std::cout << "\n上限A = 1000 / " << busy[culprit] << "ms（" << nm[culprit] << "） = "
              << (1000.0 / busy[culprit]) << " FPS\n"
              << "上限B = 1000 x " << std::setprecision(2) << eff << std::setprecision(1)
              << " / " << CPU_TOTAL << "ms = " << (1000.0 * eff / CPU_TOTAL) << " FPS\n"
              << "実測  " << (N / sec) << " FPS\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread -O2 ans10a.cpp -o ans10a && ./ans10a 30

```
Show = 30ms（10枚に1枚は 60ms）

     正味  取り出し待ち  入れ待ち   段
    13.2ms        0.0ms      9.8ms   Read
    14.2ms        0.3ms     14.3ms   Infer
    33.1ms        0.8ms      0.0ms   Show   <- 待っていない ＝ ここが犯人

上限A = 1000 / 33.1ms（Show） = 30.2 FPS
上限B = 1000 x 1.76 / 24.0ms = 73.4 FPS
実測  29.4 FPS
```

**16.7 FPS → 29.4 FPS。予測した 30.2 FPS のすぐ下です。当たりました。**

### (3) 3つの数字の読み方

- **取り出し待ちが大きい** ⇒ 上流が供給しきれていない（自分は飢えている）
- **入れ待ちが大きい** ⇒ 下流が詰まっている（押し戻されている）
- **どちらも小さい** ⇒ **この段が全力で走っている ＝ 犯人**

> **ボトルネックとは「待っていない唯一の段」のこと。**

表を見ると Read は入れ待ち 9.8ms、Infer は入れ待ち 14.3ms、
**Show だけがどちらも 1ms 以下**です。

そして **犯人を境に、上流は「入れ待ち」、下流は「取り出し待ち」**にきれいに分かれます。
いまは犯人が最下流なので、上流2つとも入れ待ちになりました。
**待ち時間の向きが、犯人の位置を指しています。**

### (4) 「正味が大きい段が犯人」ではない

Read（13.2ms）と Infer（14.2ms）を比べると Infer のほうが大きいですが、
**どちらも犯人ではありません。** 両方とも入れ待ちで止まっているからです。

**見るべきは正味の大小ではなく、待っているかどうか**です。
これを取り違えると、まったく効かない改善に時間を使うことになります。

（キューの長さを見ても同じことが分かります（演習6）。
ただしキューの長さは「今この瞬間」の値なので、
**時間を足し込んだこの3つの数字のほうが確かです。**）

## 発展課題2 の解答 ―― 犯人の段を軽くしたら

`ans10a.cpp` は第1引数で Show の時間を変えられます。**作り直す必要はありません。**

In [ ]:
!./ans10a 4

```
Show = 4ms（10枚に1枚は 8ms）

     正味  取り出し待ち  入れ待ち   段
    15.4ms        0.0ms      0.0ms   Read   <- 待っていない ＝ ここが犯人
    15.2ms        0.6ms      0.0ms   Infer
     4.6ms       11.3ms      0.0ms   Show

上限A = 1000 / 15.4ms（Read） = 64.9 FPS
上限B = 1000 x 1.74 / 24.0ms = 72.5 FPS
実測  62.5 FPS
```

### (1) 29.4 FPS -> 62.5 FPS

`show` の 33.1ms が 4.6ms になったので、上限A が 30.2 → 64.9 FPS に上がりました。

**隠すのではなく、仕事そのものを減らしたので、上限A が動きました。**
これは 10-4-2 の「計算が大きい段は減らすしかない」の裏返しです。
**待ちの段でも、減らせるなら減らしたほうが効きます。**

### (2) 犯人は Read に移った

Show は今度は**取り出し待ち 11.3ms**。飢えています。
そして Read が、どちらも待たない段になりました。

演習1・7で見た**ボトルネックの移動**が、そのまま数字に出ています。

> **1回直したら、必ず測り直す。犯人はもうそこにいない。**

### (3) そして、次はもう伸びない

```
上限A = 64.9 FPS
上限B = 72.5 FPS      <- 近づいてきた
実測  62.5 FPS
```

Read（13ms）と Infer（11ms）は**どちらも計算する段**です。
2つとも同時に走らせるには、コアが2つぶん要ります。

つまり **ここから先は、Read を2人にしても上限B が天井になります**（演習7-2-3）。
できるのは「デコードを軽くする」「入力を小さくする」といった、
**計算そのものを減らす**改造だけです。

**コアの少ないマシンでは、上限B がもっと下（たとえば 40 FPS）に来ます。**
そのときは実測が上限A（64.9）にまるで届かないはずです。自分の結果で確かめてください。

---

## 参考：本番での手順

1. **まず直列版で測る。** 配布される直列版のプロファイリング用コードを使ってください
2. **「計測外」がゼロに近いことを確かめる。** ここが大きいうちは、どの数字も信用しない
3. **一番大きい段と、その種類（計算／待ち）を見る**
4. **上限A と 上限B のどちらが効いているかを計算する**
5. **決めて、組んで、段ごとに 正味 / 取り出し待ち / 入れ待ち を測って確かめる**
6. **また 1 に戻る。** ボトルネックは移動しています

測るときの約束です。

- 数回まわして、最小値と中央値を見る。1回目は捨てる
- FPS だけでなく、レイテンシとキューの長さも見る
- 表示（`imshow`、デバッグ `cout`）の有無を変えたら、**全部測り直す**

**「速くなった気がする」は、3日間でいちばん高くつく言葉です。**